### RAG Pipelines - Data Ingestion to Vector DB Pipeline

In [13]:
import os
from pathlib import Path
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [14]:
### Read all the pdfs inside the directory
# Go through the ../data folder, find every PDF inside it (including subfolders), read each PDF, 
# and combine all the pages into one list

def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

all_pdf_documents = process_all_pdfs("../data")

Found 3 PDF files to process

Processing: pdf1.pdf
  ✓ Loaded 88 pages

Processing: pdf2.pdf
  ✓ Loaded 36 pages

Processing: pdf3.pdf
  ✓ Loaded 5 pages

Total documents loaded: 129


In [15]:
all_pdf_documents

[Document(metadata={'producer': 'Acrobat Distiller 4.0 for Windows', 'creator': 'Adobe PageMaker 6.52', 'creationdate': '2001-01-03T10:10:11+00:00', 'subject': 'cover', 'keywords': '', 'author': 'tana', 'title': 'cover', 'moddate': '2003-02-19T12:59:16-07:00', 'source': '..\\data\\pdfs\\pdf1.pdf', 'total_pages': 88, 'page': 0, 'page_label': '1', 'source_file': 'pdf1.pdf', 'file_type': 'pdf'}, page_content='CATTLE\nmanagement manual\nA reference for agency staff, department employees, loan officers,\nand others with a need to consult cattle management information\nDouglas Reynolds, James Waggoner, and Michael Smith\nNoNoNoNoNovvvvvember 2000ember 2000ember 2000ember 2000ember 2000\nMP-97MP-97MP-97MP-97MP-97'),
 Document(metadata={'producer': 'Acrobat Distiller 4.0 for Windows', 'creator': 'Adobe PageMaker 6.52', 'creationdate': '2001-01-03T10:10:11+00:00', 'subject': 'cover', 'keywords': '', 'author': 'tana', 'title': 'cover', 'moddate': '2003-02-19T12:59:16-07:00', 'source': '..\\data\

In [16]:
# Text Splitting - get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """ Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function = len,
        separators= ["\n\n","\n"," ",""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show examples of a chunk 
    if split_docs:
        print(f"\n Example Chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs



In [18]:
chunks = split_documents(all_pdf_documents)
chunks

Split 129 documents into 422 chunks

 Example Chunk:
Content: CATTLE
management manual
A reference for agency staff, department employees, loan officers,
and others with a need to consult cattle management information
Douglas Reynolds, James Waggoner, and Michae...
Metadata: {'producer': 'Acrobat Distiller 4.0 for Windows', 'creator': 'Adobe PageMaker 6.52', 'creationdate': '2001-01-03T10:10:11+00:00', 'subject': 'cover', 'keywords': '', 'author': 'tana', 'title': 'cover', 'moddate': '2003-02-19T12:59:16-07:00', 'source': '..\\data\\pdfs\\pdf1.pdf', 'total_pages': 88, 'page': 0, 'page_label': '1', 'source_file': 'pdf1.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Acrobat Distiller 4.0 for Windows', 'creator': 'Adobe PageMaker 6.52', 'creationdate': '2001-01-03T10:10:11+00:00', 'subject': 'cover', 'keywords': '', 'author': 'tana', 'title': 'cover', 'moddate': '2003-02-19T12:59:16-07:00', 'source': '..\\data\\pdfs\\pdf1.pdf', 'total_pages': 88, 'page': 0, 'page_label': '1', 'source_file': 'pdf1.pdf', 'file_type': 'pdf'}, page_content='CATTLE\nmanagement manual\nA reference for agency staff, department employees, loan officers,\nand others with a need to consult cattle management information\nDouglas Reynolds, James Waggoner, and Michael Smith\nNoNoNoNoNovvvvvember 2000ember 2000ember 2000ember 2000ember 2000\nMP-97MP-97MP-97MP-97MP-97'),
 Document(metadata={'producer': 'Acrobat Distiller 4.0 for Windows', 'creator': 'Adobe PageMaker 6.52', 'creationdate': '2001-01-03T10:10:11+00:00', 'subject': 'cover', 'keywords': '', 'author': 'tana', 'title': 'cover', 'moddate': '2003-02-19T12:59:16-07:00', 'source': '..\\data\

### Embedding & VectorStoreDB

In [19]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


d:\Courses\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
    # takes text and returns numpy array
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


d:\Courses\RAG\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sujal\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4744.48it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\sujal\AppData\Local\Temp\ipykernel_16028\2964522620.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### VectorStore


In [ ]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_texct
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [22]:
chunks

[Document(metadata={'producer': 'Acrobat Distiller 4.0 for Windows', 'creator': 'Adobe PageMaker 6.52', 'creationdate': '2001-01-03T10:10:11+00:00', 'subject': 'cover', 'keywords': '', 'author': 'tana', 'title': 'cover', 'moddate': '2003-02-19T12:59:16-07:00', 'source': '..\\data\\pdfs\\pdf1.pdf', 'total_pages': 88, 'page': 0, 'page_label': '1', 'source_file': 'pdf1.pdf', 'file_type': 'pdf'}, page_content='CATTLE\nmanagement manual\nA reference for agency staff, department employees, loan officers,\nand others with a need to consult cattle management information\nDouglas Reynolds, James Waggoner, and Michael Smith\nNoNoNoNoNovvvvvember 2000ember 2000ember 2000ember 2000ember 2000\nMP-97MP-97MP-97MP-97MP-97'),
 Document(metadata={'producer': 'Acrobat Distiller 4.0 for Windows', 'creator': 'Adobe PageMaker 6.52', 'creationdate': '2001-01-03T10:10:11+00:00', 'subject': 'cover', 'keywords': '', 'author': 'tana', 'title': 'cover', 'moddate': '2003-02-19T12:59:16-07:00', 'source': '..\\data\

In [25]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector database
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 422 texts...


Batches: 100%|██████████| 14/14 [00:17<00:00,  1.24s/it]


Generated embeddings with shape: (422, 384)
Adding 422 documents to vector store...
Successfully added 422 documents to vector store
Total documents in collection: 844


In [24]:
embeddings

array([[-0.04792456, -0.0802181 , -0.06860553, ...,  0.01190382,
        -0.01674122,  0.0441368 ],
       [-0.03959851, -0.02559048,  0.02100195, ..., -0.11022977,
         0.02134908, -0.02956995],
       [-0.05141967, -0.02275178, -0.04843166, ..., -0.1065855 ,
         0.07657097,  0.0038106 ],
       ...,
       [ 0.0679884 ,  0.00165483, -0.00504928, ..., -0.05478019,
        -0.0459112 ,  0.03141271],
       [ 0.01895051, -0.07737319,  0.04400429, ..., -0.05921083,
        -0.04364189,  0.02857071],
       [-0.01354858,  0.09543742,  0.01379715, ..., -0.00511278,
         0.01003051,  0.01734214]], shape=(422, 384), dtype=float32)